# Patient-Level Data Splitting
Loads the image metadata produced by `00_eda.ipynb` and builds the frozen patient-level train/val/test split.


In [1]:
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

df = pd.read_csv("../data/raw/image_metadata.csv")
print(f"Loaded {len(df)} rows from image_metadata.csv")
df.head()

Loaded 27558 rows from image_metadata.csv


,filename,path,category,height,width,aspect_ratio,total_pixels
0,C118P79ThinF_IMG_20151002_105018_cell_150.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,115,112,0.973913,12880
1,C189P150ThinF_IMG_20151203_142224_cell_84.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,121,118,0.975207,14278
2,C91P52ThinF_IMG_20150821_123116_cell_189.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,169,148,0.875740,25012
3,C84P45ThinF_IMG_20150818_101226_cell_98.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,106,136,1.283019,14416
4,C144P105ThinF_IMG_20151015_163432_cell_310.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,91,103,1.131868,9373


# # 6. Patient-Level Data Splitting

In [2]:
# Inspect the raw NIH patient-to-cell mapping structure
raw_uninfected = pd.read_csv(
    "../data/raw/patientid_cellmapping_uninfected.csv",
    header=None
)

raw_parasitized = pd.read_csv(
    "../data/raw/patientid_cellmapping_parasitized.csv",
    header=None
)

print("Raw uninfected shape:", raw_uninfected.shape)
print("Raw parasitized shape:", raw_parasitized.shape)

display(raw_uninfected.iloc[:5, :8])
display(raw_parasitized.iloc[:5, :8])

Raw uninfected shape: (201, 78)
Raw parasitized shape: (151, 634)


,0,1,2,3,4,5,6,7
0,C100P61ThinF,['C100P61ThinF_IMG_20150918_144104_cell_128.png','C100P61ThinF_IMG_20150918_144104_cell_131.png','C100P61ThinF_IMG_20150918_144104_cell_144.png','C100P61ThinF_IMG_20150918_144104_cell_21.png','C100P61ThinF_IMG_20150918_144104_cell_25.png','C100P61ThinF_IMG_20150918_144104_cell_34.png','C100P61ThinF_IMG_20150918_144104_cell_48.png'
1,C101P62ThinF,['C101P62ThinF_IMG_20150918_151006_cell_17.png','C101P62ThinF_IMG_20150918_151006_cell_29.png','C101P62ThinF_IMG_20150918_151006_cell_32.png','C101P62ThinF_IMG_20150918_151006_cell_53.png','C101P62ThinF_IMG_20150918_151006_cell_56.png','C101P62ThinF_IMG_20150918_151006_cell_59.png','C101P62ThinF_IMG_20150918_151006_cell_8.png'
2,C102P63ThinF,['C102P63ThinF_IMG_20150918_161508_cell_101.png','C102P63ThinF_IMG_20150918_161508_cell_113.png','C102P63ThinF_IMG_20150918_161508_cell_133.png','C102P63ThinF_IMG_20150918_161508_cell_14.png','C102P63ThinF_IMG_20150918_161508_cell_143.png','C102P63ThinF_IMG_20150918_161508_cell_150.png','C102P63ThinF_IMG_20150918_161508_cell_172.png'
3,C103P64ThinF,['C103P64ThinF_IMG_20150918_164250_cell_121.png','C103P64ThinF_IMG_20150918_164250_cell_128.png','C103P64ThinF_IMG_20150918_164250_cell_15.png','C103P64ThinF_IMG_20150918_164250_cell_153.png','C103P64ThinF_IMG_20150918_164250_cell_164.png','C103P64ThinF_IMG_20150918_164250_cell_19.png','C103P64ThinF_IMG_20150918_164250_cell_28.png'
4,C104P65ThinF,['C104P65ThinF_IMG_20150918_170850_cell_140.png','C104P65ThinF_IMG_20150918_170850_cell_167.png','C104P65ThinF_IMG_20150918_170850_cell_174.png','C104P65ThinF_IMG_20150918_170850_cell_189.png','C104P65ThinF_IMG_20150918_170850_cell_29.png','C104P65ThinF_IMG_20150918_170850_cell_32.png','C104P65ThinF_IMG_20150918_170850_cell_66.png'


,0,1,2,3,4,5,6,7
0,C100P61ThinF,['C100P61ThinF_IMG_20150918_144104_cell_162.png','C100P61ThinF_IMG_20150918_144104_cell_163.png','C100P61ThinF_IMG_20150918_144104_cell_164.png','C100P61ThinF_IMG_20150918_144104_cell_165.png','C100P61ThinF_IMG_20150918_144104_cell_166.png','C100P61ThinF_IMG_20150918_144104_cell_167.png','C100P61ThinF_IMG_20150918_144104_cell_168.png'
1,C101P62ThinF,['C101P62ThinF_IMG_20150918_151006_cell_61.png','C101P62ThinF_IMG_20150918_151006_cell_62.png','C101P62ThinF_IMG_20150918_151006_cell_63.png','C101P62ThinF_IMG_20150918_151006_cell_64.png','C101P62ThinF_IMG_20150918_151006_cell_65.png','C101P62ThinF_IMG_20150918_151006_cell_66.png','C101P62ThinF_IMG_20150918_151006_cell_67.png'
2,C102P63ThinF,['C102P63ThinF_IMG_20150918_161508_cell_190.png','C102P63ThinF_IMG_20150918_161508_cell_191.png','C102P63ThinF_IMG_20150918_161508_cell_192.png','C102P63ThinF_IMG_20150918_161508_cell_193.png','C102P63ThinF_IMG_20150918_161508_cell_194.png','C102P63ThinF_IMG_20150918_161508_cell_195.png','C102P63ThinF_IMG_20150918_161826_cell_170.png'
3,C103P64ThinF,['C103P64ThinF_IMG_20150918_164250_cell_170.png','C103P64ThinF_IMG_20150918_164250_cell_171.png','C103P64ThinF_IMG_20150918_164250_cell_172.png','C103P64ThinF_IMG_20150918_164250_cell_173.png','C103P64ThinF_IMG_20150918_164250_cell_174.png','C103P64ThinF_IMG_20150918_164250_cell_175.png','C103P64ThinF_IMG_20150918_164331_cell_184.png'
4,C104P65ThinF,['C104P65ThinF_IMG_20150918_170850_cell_204.png','C104P65ThinF_IMG_20150918_171154_cell_212.png','C104P65ThinF_IMG_20150918_171154_cell_213.png','C104P65ThinF_IMG_20150918_171751_cell_186.png','C104P65ThinF_IMG_20150918_171751_cell_187.png','C104P65ThinF_IMG_20150918_172053_cell_187.png','C104P65ThinF_IMG_20150918_172537_cell_149.png'


In [3]:
# Convert the wide NIH mapping files into one row per cell image
def reshape_patient_mapping(raw_df, category):
    rows = []

    for _, row in raw_df.iterrows():
        patient_id = row.iloc[0]

        # Remaining columns contain image filenames.
        # Some cells are empty because patients have different numbers of images.
        filenames = row.iloc[1:].dropna()

        for filename in filenames:
            filename = str(filename).strip()

            # Remove brackets/quotes left by the original CSV formatting
            filename = filename.strip("[]'\" ")

            if filename:
                rows.append({
                    "filename": filename,
                    "patient_id": str(patient_id).strip(),
                    "category": category
                })

    return pd.DataFrame(rows)


uninfected_patient_map = reshape_patient_mapping(
    raw_uninfected,
    "Uninfected"
)

parasitized_patient_map = reshape_patient_mapping(
    raw_parasitized,
    "Parasitized"
)

patient_map = pd.concat(
    [uninfected_patient_map, parasitized_patient_map],
    ignore_index=True
)

print("Mapped uninfected images:", len(uninfected_patient_map))
print("Mapped parasitized images:", len(parasitized_patient_map))
print("Total mapped images:", len(patient_map))
print("Unique patient IDs:", patient_map["patient_id"].nunique())

display(patient_map.head())

Mapped uninfected images: 13779
Mapped parasitized images: 13779
Total mapped images: 27558
Unique patient IDs: 201


,filename,patient_id,category
0,C100P61ThinF_IMG_20150918_144104_cell_128.png,C100P61ThinF,Uninfected
1,C100P61ThinF_IMG_20150918_144104_cell_131.png,C100P61ThinF,Uninfected
2,C100P61ThinF_IMG_20150918_144104_cell_144.png,C100P61ThinF,Uninfected
3,C100P61ThinF_IMG_20150918_144104_cell_21.png,C100P61ThinF,Uninfected
4,C100P61ThinF_IMG_20150918_144104_cell_25.png,C100P61ThinF,Uninfected


In [4]:
# Verify that NIH mapping filenames match the Kaggle dataset filenames

dataset_filenames = set(df["filename"].astype(str))
mapping_filenames = set(patient_map["filename"].astype(str))

matched_filenames = dataset_filenames.intersection(mapping_filenames)
missing_from_mapping = dataset_filenames - mapping_filenames
missing_from_dataset = mapping_filenames - dataset_filenames

print("Dataset filenames:", len(dataset_filenames))
print("Mapping filenames:", len(mapping_filenames))
print("Matched filenames:", len(matched_filenames))
print("Dataset images missing from NIH mapping:", len(missing_from_mapping))
print("NIH mapping images missing from dataset:", len(missing_from_dataset))

print("\nMatch rate:",
      f"{len(matched_filenames) / len(dataset_filenames) * 100:.2f}%")

Dataset filenames: 27558
Mapping filenames: 27558
Matched filenames: 27558
Dataset images missing from NIH mapping: 0
NIH mapping images missing from dataset: 0

Match rate: 100.00%


In [5]:
# Add patient IDs to the Kaggle dataset metadata
df_with_patients = df.merge(
    patient_map[["filename", "patient_id", "category"]],
    on="filename",
    how="left",
    suffixes=("_dataset", "_mapping")
)

# Verify that every image received a patient ID
print("Total images:", len(df_with_patients))
print("Images without patient ID:", df_with_patients["patient_id"].isna().sum())
print("Unique patients:", df_with_patients["patient_id"].nunique())

display(df_with_patients.head())

Total images: 27558
Images without patient ID: 0
Unique patients: 201


,filename,path,category_dataset,height,width,aspect_ratio,total_pixels,patient_id,category_mapping
0,C118P79ThinF_IMG_20151002_105018_cell_150.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,115,112,0.973913,12880,C118P79ThinF,Parasitized
1,C189P150ThinF_IMG_20151203_142224_cell_84.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,121,118,0.975207,14278,C189P150ThinF,Parasitized
2,C91P52ThinF_IMG_20150821_123116_cell_189.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,169,148,0.875740,25012,C91P52ThinF,Parasitized
3,C84P45ThinF_IMG_20150818_101226_cell_98.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,106,136,1.283019,14416,C84P45ThinF,Parasitized
4,C144P105ThinF_IMG_20151015_163432_cell_310.png,/Users/andersvestergaard/.cache/kagglehub/data...,Parasitized,91,103,1.131868,9373,C144P105ThinF,Parasitized


In [6]:
# Inspect patient-level class composition before splitting

patient_summary = (
    df_with_patients
    .groupby("patient_id")
    .agg(
        total_images=("filename", "count"),
        parasitized_images=(
            "category_mapping",
            lambda x: (x == "Parasitized").sum()
        ),
        uninfected_images=(
            "category_mapping",
            lambda x: (x == "Uninfected").sum()
        )
    )
    .reset_index()
)

patient_summary["patient_type"] = np.select(
    [
        (patient_summary["parasitized_images"] > 0) &
        (patient_summary["uninfected_images"] > 0),

        patient_summary["parasitized_images"] > 0,

        patient_summary["uninfected_images"] > 0
    ],
    [
        "Both",
        "Parasitized only",
        "Uninfected only"
    ],
    default="Unknown"
)

print("Number of patients:", len(patient_summary))
print("\nPatient type counts:")
print(patient_summary["patient_type"].value_counts())

print("\nImages per patient:")
print(patient_summary["total_images"].describe())

display(patient_summary.head(10))

Number of patients: 201

Patient type counts:
patient_type
Both               151
Uninfected only     50
Name: count, dtype: int64

Images per patient:
count    201.000000
mean     137.104478
std      126.789448
min       65.000000
25%       70.000000
50%       85.000000
75%      130.000000
max      702.000000
Name: total_images, dtype: float64


,patient_id,total_images,parasitized_images,uninfected_images,patient_type
0,C100P61ThinF,125,57,68,Both
1,C101P62ThinF,385,314,71,Both
2,C102P63ThinF,103,35,68,Both
3,C103P64ThinF,107,39,68,Both
4,C104P65ThinF,82,10,72,Both
5,C105P66ThinF,85,17,68,Both
6,C106P67ThinF,79,13,66,Both
7,C107P68ThinF,132,62,70,Both
8,C108P69ThinF,89,20,69,Both
9,C109P70ThinF,87,19,68,Both


In [7]:
# Create reproducible patient-level train/validation/test splits
# Stratify by patient type so "Both" and "Uninfected only"
# patients are represented across all splits.

from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temporary
train_patients, temp_patients = train_test_split(
    patient_summary,
    test_size=0.30,
    random_state=SEED,
    stratify=patient_summary["patient_type"]
)

# Second split: divide temporary set equally into validation and test
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_patients["patient_type"]
)

train_ids = set(train_patients["patient_id"])
val_ids = set(val_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

train_df = df_with_patients[
    df_with_patients["patient_id"].isin(train_ids)
].copy()

val_df = df_with_patients[
    df_with_patients["patient_id"].isin(val_ids)
].copy()

test_df = df_with_patients[
    df_with_patients["patient_id"].isin(test_ids)
].copy()

print("PATIENT COUNTS")
print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\nPATIENT TYPE DISTRIBUTION")
print("\nTrain:")
print(train_patients["patient_type"].value_counts())
print("\nValidation:")
print(val_patients["patient_type"].value_counts())
print("\nTest:")
print(test_patients["patient_type"].value_counts())

print("\nIMAGE COUNTS")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

PATIENT COUNTS
Train: 140
Validation: 30
Test: 31

PATIENT TYPE DISTRIBUTION

Train:
patient_type
Both               105
Uninfected only     35
Name: count, dtype: int64

Validation:
patient_type
Both               23
Uninfected only     7
Name: count, dtype: int64

Test:
patient_type
Both               23
Uninfected only     8
Name: count, dtype: int64

IMAGE COUNTS
Train: 19124
Validation: 4324
Test: 4110
Total: 27558


In [8]:
# Verify that there is no patient leakage between splits

train_patient_ids = set(train_df["patient_id"])
val_patient_ids = set(val_df["patient_id"])
test_patient_ids = set(test_df["patient_id"])

train_val_overlap = train_patient_ids.intersection(val_patient_ids)
train_test_overlap = train_patient_ids.intersection(test_patient_ids)
val_test_overlap = val_patient_ids.intersection(test_patient_ids)

print("PATIENT OVERLAP CHECK")
print("Train vs Validation:", len(train_val_overlap))
print("Train vs Test:", len(train_test_overlap))
print("Validation vs Test:", len(val_test_overlap))

assert len(train_val_overlap) == 0, "Patient leakage between train and validation!"
assert len(train_test_overlap) == 0, "Patient leakage between train and test!"
assert len(val_test_overlap) == 0, "Patient leakage between validation and test!"

print("\nNo patient leakage detected.")

PATIENT OVERLAP CHECK
Train vs Validation: 0
Train vs Test: 0
Validation vs Test: 0

No patient leakage detected.


In [9]:
# Check class distribution across train, validation, and test sets

def show_class_distribution(split_name, split_df):
    counts = split_df["category_dataset"].value_counts()
    percentages = split_df["category_dataset"].value_counts(normalize=True) * 100

    print(f"\n{split_name}")
    print("-" * len(split_name))

    for category in counts.index:
        print(
            f"{category}: {counts[category]} images "
            f"({percentages[category]:.2f}%)"
        )

show_class_distribution("TRAIN", train_df)
show_class_distribution("VALIDATION", val_df)
show_class_distribution("TEST", test_df)


TRAIN
-----
Uninfected: 9576 images (50.07%)
Parasitized: 9548 images (49.93%)

VALIDATION
----------
Parasitized: 2264 images (52.36%)
Uninfected: 2060 images (47.64%)

TEST
----
Uninfected: 2143 images (52.14%)
Parasitized: 1967 images (47.86%)


In [10]:
# Final split integrity checks

train_files = set(train_df["filename"])
val_files = set(val_df["filename"])
test_files = set(test_df["filename"])

print("SPLIT INTEGRITY CHECK")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Total images:", len(train_df) + len(val_df) + len(test_df))

print("\nIMAGE OVERLAP")
print("Train vs Validation:", len(train_files & val_files))
print("Train vs Test:", len(train_files & test_files))
print("Validation vs Test:", len(val_files & test_files))

assert len(train_df) + len(val_df) + len(test_df) == len(df_with_patients)
assert len(train_files & val_files) == 0
assert len(train_files & test_files) == 0
assert len(val_files & test_files) == 0

print("\nAll images assigned exactly once with no overlap.")

SPLIT INTEGRITY CHECK
Train images: 19124
Validation images: 4324
Test images: 4110
Total images: 27558

IMAGE OVERLAP
Train vs Validation: 0
Train vs Test: 0
Validation vs Test: 0

All images assigned exactly once with no overlap.


# 7. Export Frozen Split Manifest
Persist the patient-level split so every other notebook loads the same frozen split instead of recomputing it.

In [11]:
# Build and save a single manifest: one row per image, with its assigned split.
import os

split_frames = [
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    test_df.assign(split="test"),
]

manifest = pd.concat(split_frames, ignore_index=True)[
    ["filename", "patient_id", "category_dataset", "split"]
].rename(columns={"category_dataset": "category"})

os.makedirs("../data/splits", exist_ok=True)
manifest_path = "../data/splits/patient_split.csv"
manifest.to_csv(manifest_path, index=False)

print(f"Wrote {len(manifest)} rows to {manifest_path}")
print(manifest["split"].value_counts())
manifest.head()

Wrote 27558 rows to ../data/splits/patient_split.csv
split
train    19124
val       4324
test      4110
Name: count, dtype: int64


,filename,patient_id,category,split
0,C118P79ThinF_IMG_20151002_105018_cell_150.png,C118P79ThinF,Parasitized,train
1,C189P150ThinF_IMG_20151203_142224_cell_84.png,C189P150ThinF,Parasitized,train
2,C91P52ThinF_IMG_20150821_123116_cell_189.png,C91P52ThinF,Parasitized,train
3,C84P45ThinF_IMG_20150818_101226_cell_98.png,C84P45ThinF,Parasitized,train
4,C144P105ThinF_IMG_20151015_163432_cell_310.png,C144P105ThinF,Parasitized,train
